In [ ]:
# --- imports block for latitudinal binning and mapping ---
import gc
import os

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import xarray as xr

print("libraries loaded")

In [ ]:


# local directory search paths (nc files downlaoded from climatology script)
search_paths = [
    "fixed Arabian_Sea chl bioargo",
    "fixed Bay_of_Bengal chl bioargo"
]

standard_depths = np.arange(0, 305, 5)
bins = np.arange(4, 22, 2)
labels = [f"{bins[i]}-{bins[i+1]}°N" for i in range(len(bins)-1)]

# initialize the final dataframe structure with explicit default arrays
final_climatology_data = {"Depth (m)": standard_depths}
for region in ["Arabian_Sea", "Bay_of_Bengal"]:
    for bin_label in labels:
        clean_col = f"{region}_{bin_label.replace('°N', 'N')}"
        final_climatology_data[clean_col] = np.zeros_like(standard_depths)

# file path accumulator dictionary
regional_profile_accumulator = {
    "Arabian Sea": {label: [] for label in labels},
    "Bay of Bengal": {label: [] for label in labels}
}

print("mapping local NC files back to geographic bins...")
seen_wmos = set()

for path in search_paths:
    if not os.path.exists(path):
        print(f"whoops path layout missing: {path}?? check your folder root.")
        continue
    for file in os.listdir(path):
        if not file.endswith(".nc"):
            continue
        try:
            filename = os.path.basename(file)
            clean_name = filename.replace("BR", "").replace("BD", "").replace("R", "").replace("D", "")
            wmo_id = clean_name.split("_")[0]

            if wmo_id in seen_wmos:
                continue

            full_file_path = os.path.join(path, file)
            with xr.open_dataset(full_file_path, decode_times=False) as ds:
                lats = ds['LATITUDE'].values
                longs = ds['LONGITUDE'].values
                valid_mask = (~np.isnan(lats)) & (~np.isnan(longs)) & (np.abs(lats) <= 90) & (np.abs(longs) <= 180)

                if np.any(valid_mask):
                    median_lat = np.median(lats[valid_mask])
                    median_lon = np.median(longs[valid_mask])

                    if 5.0 <= median_lat <= 20.0:
                        if 45.0 <= median_lon <= 77.5:
                            true_region = "Arabian Sea"
                        elif 77.5 < median_lon <= 100.0:
                            true_region = "Bay of Bengal"
                        else:
                            continue

                        bin_idx = np.digitize(median_lat, bins) - 1
                        if 0 <= bin_idx < len(labels):
                            target_label = labels[bin_idx]
                            regional_profile_accumulator[true_region][target_label].append(full_file_path)
                            seen_wmos.add(wmo_id)
        except Exception:
            continue

print("processing vertical columns with QC Flag 1 & 2 verification...")

for region in ["Arabian Sea", "Bay of Bengal"]:
    for bin_label in labels:
        file_paths = regional_profile_accumulator[region][bin_label]
        if not file_paths:
            continue

        bin_profiles = []
        for path in file_paths:
            try:
                with xr.open_dataset(path, decode_times=False) as ds:
                    if 'CHLA_ADJUSTED' in ds.variables:
                        chl_var, pres_var, qc_var = 'CHLA_ADJUSTED', 'PRES_ADJUSTED', 'CHLA_ADJUSTED_QC'
                    else:
                        chl_var, pres_var, qc_var = 'CHLA', 'PRES', 'CHLA_QC'

                    if chl_var in ds.variables and pres_var in ds.variables and qc_var in ds.variables:
                        raw_chl = ds[chl_var].values.flatten()
                        raw_pres = ds[pres_var].values.flatten()
                        raw_qc = ds[qc_var].values.astype(str).flatten()

                        # filter to keep strictly QC 1 (good) and QC 2 (probably good)
                        qc_mask = (raw_qc == '1') | (raw_qc == '2')
                        valid_data = qc_mask & (~np.isnan(raw_chl)) & (~np.isnan(raw_pres)) & \
                                     (raw_pres >= 0) & (raw_pres <= 350) & (raw_chl >= 0)

                        if np.sum(valid_data) > 3:
                            p_filtered = raw_pres[valid_data]
                            c_filtered = raw_chl[valid_data]

                            sort_idx = np.argsort(p_filtered)
                            interp_chl = np.interp(
                                standard_depths,
                                p_filtered[sort_idx],
                                c_filtered[sort_idx],
                                left=np.nan, right=np.nan
                            )
                            bin_profiles.append(interp_chl)
            except Exception:
                continue

        if bin_profiles:
            mean_profile = np.nanmean(np.array(bin_profiles), axis=0)
            col_name = f"{region.replace(' ', '_')}_{bin_label.replace('°N', 'N')}"
            final_climatology_data[col_name] = mean_profile

del bin_profiles
gc.collect()

# convert compiled vectors to localized matrix sheets
df_lat_climatology = pd.DataFrame(final_climatology_data).set_index("Depth (m)")
csv_output_path = "QC_Filtered_Latitudinal_Chl_Climatology.csv"
df_lat_climatology.to_csv(csv_output_path)
print(f"yay lat binned climatology saved to: {csv_output_path}")

In [ ]:


print("generating sorted vertical layout visualization panels...")

# clean text strings out of columns to parse labels
raw_bins = list(set([col.split('_')[-1] for col in df_lat_climatology.columns]))
bin_labels = sorted(raw_bins, key=lambda x: int(x.split('-')[0]), reverse=True)

global_max_chl = df_lat_climatology.max().max()
x_max_limit = global_max_chl * 1.1 if global_max_chl > 0 else 2.0

fig, axes = plt.subplots(len(bin_labels), 2, figsize=(12, 4 * len(bin_labels)), constrained_layout=True)

if len(bin_labels) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, bin_name in enumerate(bin_labels):
    display_bin_name = bin_name.replace('N', '°N')
    target_regions = [
        ("Arabian Sea", f"Arabian_Sea_{bin_name}"),
        ("Bay of Bengal", f"Bay_of_Bengal_{bin_name}")
    ]

    for col_idx, (region_title, col_name) in enumerate(target_regions):
        ax = axes[row_idx, col_idx]

        # confirm active baseline numbers exist inside columns before graphing arrays
        if col_name in df_lat_climatology.columns and not (df_lat_climatology[col_name] == 0).all():
            y_depths = df_lat_climatology.index.values
            x_chl = df_lat_climatology[col_name].values

            ax.plot(x_chl, y_depths, color='darkgreen', linewidth=2, label='Chl-A Mean Profile')
            ax.legend(loc='lower right', fontsize=9)
        else:
            ax.text(0.5, 0.5, 'whoops no active records found??', transform=ax.transAxes,
                    ha='center', va='center', color='gray', fontsize=11, fontstyle='italic')

        ax.set_ylim(300, 0)
        ax.set_xlim(0, x_max_limit)
        ax.xaxis.set_ticks_position('top')
        ax.xaxis.set_label_position('top')
        ax.set_xlabel('Chl-A (mg/m³)', fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.6)

        if col_idx == 0:
            ax.set_ylabel('Depth (m)', fontsize=11, fontweight='bold')
            ax.text(-0.35, 0.5, f"Zone: {display_bin_name}", transform=ax.transAxes,
                    fontsize=12, fontweight='bold', rotation=90, va='center', ha='center')

        if row_idx == 0:
            ax.set_title(region_title, fontsize=14, fontweight='bold', pad=35)

plt.savefig("Corrected_Separate_Latitudinal_Profiles.png", dpi=200)
plt.show()
print("panel configurations plotted.")

In [ ]:


print("building spatial cartopy framework map layers...")

plt.figure(figsize=(13, 8))
map_projection = ccrs.PlateCarree()
ax = plt.axes(projection=map_projection)
ax.set_extent([40.0, 105.0, 2.0, 25.0], crs=map_projection)

# add physical features smoothly
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1.2, edgecolor='black', zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray', linestyle=':', zorder=2)
ax.add_feature(cfeature.OCEAN, facecolor='aliceblue', alpha=0.5, zorder=0)

# construct gridlines
gl = ax.gridlines(crs=map_projection, draw_labels=True, linewidth=1, color='gainsboro', linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlocator = mticker.FixedLocator(np.arange(40, 110, 5))
gl.ylocator = mticker.FixedLocator(np.arange(0, 30, 2))
gl.xlabel_style = {'size': 10, 'color': 'black'}
gl.ylabel_style = {'size': 10, 'color': 'black'}

# reconstruct localized float mapping coordinate parameters from structural arrays
float_mapping_list = []
seen_wmos.clear()

# establish custom color wheels for bands
color_list = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']
bin_colors = {labels[idx]: color_list[idx % len(color_list)] for idx in range(len(labels))}

for path in search_paths:
    if not os.path.exists(path):
        continue
    for file in os.listdir(path):
        if not file.endswith(".nc"):
            continue
        try:
            filename = os.path.basename(file)
            clean_name = filename.replace("BR", "").replace("BD", "").replace("R", "").replace("D", "")
            wmo_id = clean_name.split("_")[0]

            if wmo_id in seen_wmos:
                continue

            full_file_path = os.path.join(path, file)
            with xr.open_dataset(full_file_path, decode_times=False) as ds:
                lats = ds['LATITUDE'].values
                longs = ds['LONGITUDE'].values
                
                if 'CHLA_ADJUSTED' in ds.variables:
                    qc_var = 'CHLA_ADJUSTED_QC'
                elif 'CHLA' in ds.variables:
                    qc_var = 'CHLA_QC'
                else:
                    continue

                raw_qc = ds[qc_var].values.astype(str).flatten()
                valid_mask = (~np.isnan(lats)) & (~np.isnan(longs)) & (np.abs(lats) <= 90) & (np.abs(longs) <= 180)
                has_valid_qc = np.any((raw_qc == '1') | (raw_qc == '2'))

                if np.any(valid_mask) and has_valid_qc:
                    median_lat = np.median(lats[valid_mask])
                    median_lon = np.median(longs[valid_mask])

                    if 5.0 <= median_lat <= 20.0 and (45.0 <= median_lon <= 100.0):
                        bin_idx = np.digitize(median_lat, bins) - 1
                        if 0 <= bin_idx < len(labels):
                            assigned_bin = labels[bin_idx]
                            float_mapping_list.append({
                                "Longitude": median_lon, "Latitude": median_lat, "Bin": assigned_bin
                            })
                            seen_wmos.add(wmo_id)
        except Exception:
            continue

df_map_floats = pd.DataFrame(float_mapping_list)

# plot individual points mapping back to matched band configurations
for bin_name in labels:
    bin_subset = df_map_floats[df_map_floats['Bin'] == bin_name]
    if not bin_subset.empty:
        ax.scatter(
            bin_subset['Longitude'].values, bin_subset['Latitude'].values,
            color=bin_colors[bin_name], edgecolor='black', s=80, alpha=0.9,
            label=f"Zone: {bin_name} (n={len(bin_subset)})", transform=map_projection, zorder=3
        )

# demarcate boundaries
ax.axhline(y=5.0, color='crimson', linestyle='-.', linewidth=1.5, alpha=0.7, zorder=2)
ax.axhline(y=20.0, color='crimson', linestyle='-.', linewidth=1.5, alpha=0.7, zorder=2)

plt.title("Spatial Distribution of BGC-Argo Floats\nGrouped by 2-Degree Latitudinal Bands", fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='lower left', framealpha=0.95, fontsize=10, title="Latitudinal Bands")

plt.savefig("Core_Basin_Float_Map.png", bbox_inches='tight', dpi=200)
plt.show()
print(f"tracking map generated! floats plotted: {len(df_map_floats)}")